# Diamond Price Prediction — Exploratory Notebook

**Role of this notebook.** This is the *data scientist's* artifact: it takes raw data and
produces a trained model, the way exploration actually happens — ingestion, cleaning, EDA,
a model bake-off, tuning, and evaluation, all in one place.

**What is deliberately *not* here.** No Docker, no API, no CI/CD, no monitoring. Operations
do **not** belong in a notebook. This notebook stops at a trained model + its metrics — the
*handoff point*. Everything after that lives in the production project (`../src/`, `../api/`, …),
which is what we build by refactoring this notebook in Module 2.

> Environment: run in WSL2/Ubuntu (or macOS/Linux) per Module 0.

In [ ]:
# Core data-science stack (no ops libraries here)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

SEED = 42
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Imports ready.")

## 1. Data Ingestion & Loading

We load from a **versioned local file** (`../data/raw/diamonds.csv`) rather than a network fetch — so every run is reproducible and the data can be DVC-tracked later.

In [ ]:
df = pd.read_csv("../data/raw/diamonds.csv")
print("Shape:", df.shape)
display(df.head())
print(df.dtypes)
df.describe()

## 2. Data Cleaning

Check for missing values and duplicates, and remove physically impossible rows: `x`, `y`, `z` are diamond dimensions in mm, so a value of 0 is a data error.

In [ ]:
print("Missing values:\n", df.isnull().sum().to_string())
print("\nDuplicate rows:", df.duplicated().sum())

before = len(df)
df = df.drop_duplicates()
df = df[(df[["x", "y", "z"]] > 0).all(axis=1)].reset_index(drop=True)
print(f"Removed {before - len(df)} rows (duplicates + zero-dimension). Remaining: {len(df)}")

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["price"], bins=50, color="steelblue", edgecolor="black")
axes[0].set_title("Price distribution"); axes[0].set_xlabel("Price ($)")
axes[1].hist(np.log1p(df["price"]), bins=50, color="seagreen", edgecolor="black")
axes[1].set_title("log(1+Price) — much closer to normal"); axes[1].set_xlabel("log price")
plt.tight_layout(); plt.show()
print("Price is right-skewed; carat is expected to be the dominant driver.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].scatter(df["carat"], df["price"], alpha=0.3, s=8)
axes[0, 0].set_title("Carat vs Price"); axes[0, 0].set_xlabel("carat"); axes[0, 0].set_ylabel("price")
sns.boxplot(data=df, x="cut", y="price", ax=axes[0, 1]); axes[0, 1].set_title("Price by Cut")
sns.boxplot(data=df, x="color", y="price", ax=axes[1, 0]); axes[1, 0].set_title("Price by Color")
sns.boxplot(data=df, x="clarity", y="price", ax=axes[1, 1]); axes[1, 1].set_title("Price by Clarity")
plt.tight_layout(); plt.show()

In [ ]:
# Correlation among numeric features
num = ["carat", "depth", "table", "x", "y", "z", "price"]
sns.heatmap(df[num].corr(), annot=True, fmt=".2f", cmap="Blues")
plt.title("Numeric correlations"); plt.show()

**Note the ordinal features.** `cut`, `color`, and `clarity` are *ordered* categories, not
nominal ones. Encoding them alphabetically would destroy that order and mislead the model, so we
encode them in true quality order (worst → best):

- cut: Fair < Good < Very Good < Premium < Ideal
- color: J < I < H < G < F < E < D
- clarity: I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF

## 4. Feature Preparation

In [ ]:
TARGET = "price"
ordinal_features = {
    "cut": ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    "color": ["J", "I", "H", "G", "F", "E", "D"],
    "clarity": ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],
}
numeric_features = ["carat", "depth", "table", "x", "y", "z"]

preprocessor = ColumnTransformer([
    ("ord", OrdinalEncoder(categories=[ordinal_features[c] for c in ordinal_features]), list(ordinal_features)),
    ("num", StandardScaler(), numeric_features),
])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 5. Model Bake-off

We compare several regression algorithms with 3-fold cross-validation on the training set,
scoring by RMSE. Each candidate is wrapped in the same preprocessing pipeline for a fair test.

> **Why no Naive Bayes?** Naive Bayes is a *classification* algorithm — it predicts discrete
> classes, not a continuous price. It doesn't apply to a regression problem, so it's excluded.

In [ ]:
candidates = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(random_state=SEED),
    "DecisionTree": DecisionTreeRegressor(random_state=SEED),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),  # 100 trees keeps the bake-off fast; tree count is tuned below
    "GradientBoosting": GradientBoostingRegressor(random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=SEED),  # histogram-based GBM: fast on 50k+ rows, small artifact
    "KNN": KNeighborsRegressor(),
}
try:
    from xgboost import XGBRegressor
    candidates["XGBoost"] = XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0)
except ImportError:
    print("(xgboost not installed — install it to include XGBoost in the bake-off)")

results = []
for name, model in candidates.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=3,
                             scoring="neg_root_mean_squared_error", n_jobs=-1)
    results.append({"model": name, "cv_rmse": -scores.mean(), "std": scores.std()})

leaderboard = pd.DataFrame(results).sort_values("cv_rmse").reset_index(drop=True)
display(leaderboard)

In [ ]:
plt.barh(leaderboard["model"], leaderboard["cv_rmse"], color="steelblue")
plt.gca().invert_yaxis(); plt.xlabel("Cross-validated RMSE (lower is better)")
plt.title("Model bake-off"); plt.tight_layout(); plt.show()

## 6. Select & Tune the Best Model

Take the top model from the bake-off and tune its hyperparameters with a small randomized search.

In [ ]:
best_name = leaderboard.iloc[0]["model"]
print("Best model:", best_name)

param_grids = {
    "RandomForest": {"model__n_estimators": [100, 200], "model__max_depth": [None, 20, 30]},
    "GradientBoosting": {"model__n_estimators": [100, 200], "model__learning_rate": [0.05, 0.1], "model__max_depth": [2, 3]},
    "HistGradientBoosting": {"model__max_iter": [200, 400], "model__learning_rate": [0.05, 0.1], "model__max_depth": [None, 6]},
    "XGBoost": {"model__n_estimators": [200, 400], "model__learning_rate": [0.05, 0.1], "model__max_depth": [4, 6]},
    "DecisionTree": {"model__max_depth": [10, 20, None]},
    "KNN": {"model__n_neighbors": [5, 10, 15]},
}
best_pipe = Pipeline([("prep", preprocessor), ("model", candidates[best_name])])
grid = param_grids.get(best_name, {})
if grid:
    search = RandomizedSearchCV(best_pipe, grid, n_iter=5, cv=3,
                                scoring="neg_root_mean_squared_error", random_state=SEED, n_jobs=-1)
    search.fit(X_train, y_train)
    best_pipe = search.best_estimator_
    print("Best params:", search.best_params_)
else:
    best_pipe.fit(X_train, y_train)

## 7. Final Evaluation on the Holdout Set

Metrics are computed on the test set the model has **never seen** — no leakage.

In [ ]:
preds = best_pipe.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R2:   {r2:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, preds, alpha=0.3, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted"); axes[0].set_title("Predicted vs Actual")
residuals = y_test - preds
axes[1].scatter(preds, residuals, alpha=0.3, s=8); axes[1].axhline(0, color="r", ls="--")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual"); axes[1].set_title("Residuals")
plt.tight_layout(); plt.show()

### 7.1 Baseline comparison

A strong-looking score means nothing without a reference point. The tuned model must clearly beat
a plain `LinearRegression` on the *same* holdout, on all three metrics.

In [ ]:
baseline = Pipeline([("prep", preprocessor), ("model", LinearRegression())]).fit(X_train, y_train)
b_preds = baseline.predict(X_test)
b_mae = mean_absolute_error(y_test, b_preds)
b_rmse = np.sqrt(mean_squared_error(y_test, b_preds))
b_r2 = r2_score(y_test, b_preds)

comparison = pd.DataFrame({
    "model": ["LinearRegression (baseline)", f"{best_name} (tuned)"],
    "MAE": [b_mae, mae], "RMSE": [b_rmse, rmse], "R2": [b_r2, r2],
}).round(3)
display(comparison)
assert (rmse < b_rmse) and (mae < b_mae) and (r2 > b_r2), "Tuned model failed to beat the baseline!"
print(f"Improvement over baseline: RMSE -{1 - rmse/b_rmse:.1%}, MAE -{1 - mae/b_mae:.1%}")

### 7.2 Sanity checks: is this score too good?

An R² above ~0.95 should be treated as a **red flag until proven otherwise** — the usual culprits
are data leakage or overfitting. Three explicit checks before we accept it:

In [ ]:
# (1) Holdout isolation: X_test / y_test are used ONLY in section 7 -- never in fit(), CV, or the search above.
# (2) No target-derived features: every input is a physical property of the stone, known before pricing.
print("Features used:", list(X.columns))

# (3) Train/test contamination: identical feature-rows spanning the split would let the model memorise.
#     (Full duplicates were dropped in section 2; same-spec stones with different prices are label noise, not leakage.)
overlap = pd.merge(X_train.drop_duplicates(), X_test.drop_duplicates(), how="inner")
print(f"Identical feature-rows shared by train & test: {len(overlap)} of {len(X_test):,} test rows")

# (4) CV-vs-holdout gap: a healthy model scores about the same in CV as on unseen data.
cv_rmse_best = leaderboard.loc[leaderboard["model"] == best_name, "cv_rmse"].iloc[0]
print(f"Bake-off CV RMSE: {cv_rmse_best:,.0f}  |  tuned holdout RMSE: {rmse:,.0f}  (gap: {abs(cv_rmse_best - rmse):,.0f})")

**Verdict — the high R² is genuine, not leakage.** Diamond pricing is close to a deterministic
function of the 4 Cs: carat alone correlates ~0.92 with price, and cut/color/clarity plus the
physical dimensions explain most of the rest. The checks above confirm the holdout was never
touched during training or model selection, no feature encodes the target, and the CV and holdout
scores agree (no overfit gap). R² ≈ 0.98 is the well-documented, expected result on this dataset.
The MLOps lesson is not squeezing out more accuracy — it's making this result *reproducible,
testable, and deployable*, which is exactly what the production repo does.

## 8. Persist the Model Artifact — the Handoff

Save the fitted pipeline and its metrics. **This is where the notebook ends and MLOps begins.**

In [ ]:
import json, pathlib
pathlib.Path("../models").mkdir(exist_ok=True)
joblib.dump(best_pipe, "../models/diamond_pipeline.joblib")
metrics = {
    "model": best_name, "mae": float(mae), "rmse": float(rmse), "r2": float(r2),
    "baseline": {"model": "LinearRegression", "mae": float(b_mae), "rmse": float(b_rmse), "r2": float(b_r2)},
    "n_test": int(len(X_test)), "seed": SEED,
}
with open("../models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved ../models/diamond_pipeline.joblib and metrics.json")
print(metrics)

## 9. What this notebook is *not*, and what comes next

This notebook works — but it is **not production-ready**, and that's the point:

- it's **monolithic** and **stateful** (cells depend on run order and in-memory variables),
- paths and parameters are **hardcoded**,
- there are **no tests** and **no way to reproduce it** with one command,
- and it mixes exploration with the final pipeline.

In **Module 2** we refactor exactly this notebook into the modular project alongside it
(`../src/diamond/`, `../api/`, `../tests/`, `Makefile`, CI): the same logic, now reproducible,
testable, serveable, and deployable. Compare `train.py` and `preprocess.py` to sections 4–8 above —
that mapping *is* the lesson.